For MR-Eye Track study

In [ ]:
import os
import shutil
from pathlib import Path

subject_num = 15

# Define the folder path
folder_path = f"/home/debi/jaime/repos/MR-EyeTrack/data/study/sub-{subject_num:03d}"

# Extract the subject number from the folder name
subject_num = os.path.basename(folder_path).split('-')[1]  # e.g., "012" from "sub-012"

# Create the target folders
rawdata_folder = os.path.join(folder_path, 'rawdata')
dicom_folder = os.path.join(folder_path, 'dicom')
et_folder = os.path.join(folder_path, 'et')

for folder in [rawdata_folder, dicom_folder, et_folder]:
    os.makedirs(folder, exist_ok=True)
    print(f"Created/verified folder: {folder}")

# Process files recursively
for root, dirs, files in os.walk(folder_path):
    for filename in files:
        file_path = os.path.join(root, filename)
        
        # Skip files in the newly created folders
        if any(file_path.startswith(skip_folder) for skip_folder in [rawdata_folder, dicom_folder, et_folder]):
            continue
        
        new_filename = filename
        
        # Original Renaming Rules
        # Rule 1: Replace 'fixed_dot-16_grid_T1w' with 'mreyetrack_4points'
        if 'fixed_dot-16_grid_T1w' in filename:
            new_filename = filename.replace('fixed_dot-16_grid_T1w', 'mreyetrack_4points')
        
        # Rule 2: Rename .dat files
        if filename.endswith('.dat'):
            # Special case: ...AdjCoilSens.dat -> sub-XXX_AdjCoilSens.dat
            if filename.endswith('AdjCoilSens.dat'):
            new_filename = f"sub-{subject_num}_AdjCoilSens.dat"
            # General case: remove everything before 'sub' and normalize sub_ -> sub-
            elif 'sub' in filename:
            sub_index = filename.index('sub')
            new_filename = filename[sub_index:]
            new_filename = new_filename.replace('sub_', 'sub-', 1)
        
        # File Organization Rules
        # Rule 3: Move .dat files to rawdata
        if filename.endswith('.dat'):
            dest_path = os.path.join(rawdata_folder, new_filename)
            shutil.move(file_path, dest_path)
            print(f"Moved .dat: {filename} → rawdata/{new_filename}")
        
        # Rule 4: Move .dcm files to dicom and rename to sub-XXX, then remove the source folder
        elif filename.endswith('.dcm'):
            new_dcm_name = f"sub-{subject_num}.dcm"
            dest_path = os.path.join(dicom_folder, new_dcm_name)
            shutil.move(file_path, dest_path)
            print(f"Moved and renamed .dcm: {filename} → dicom/{new_dcm_name}")

            # Remove the folder where the dcm was (if safe)
            src_folder = os.path.dirname(file_path)
            protected = {rawdata_folder, dicom_folder, et_folder, folder_path}
            if src_folder not in protected:
                try:
                    os.rmdir(src_folder)
                    print(f"Removed empty folder: {src_folder}")
                except OSError:
                    # If not empty, remove recursively
                    try:
                        shutil.rmtree(src_folder)
                        print(f"Removed folder and its contents: {src_folder}")
                    except Exception as e:
                        print(f"Could not remove folder {src_folder}: {e}")
        
        # Rule 5: Move other files to et folder
        elif filename.endswith(('.csv', '.EDF', '.log', '.psydat')):
            dest_path = os.path.join(et_folder, new_filename)
            shutil.move(file_path, dest_path)
            if new_filename != filename:
                print(f"Moved and renamed: {filename} → et/{new_filename}")
            else:
                print(f"Moved: {filename} → et/")

print("\nFile organization and renaming completed!")


Created/verified folder: /home/debi/jaime/repos/MR-EyeTrack/data/study/sub-015/rawdata
Created/verified folder: /home/debi/jaime/repos/MR-EyeTrack/data/study/sub-015/dicom
Created/verified folder: /home/debi/jaime/repos/MR-EyeTrack/data/study/sub-015/et
Moved .dat: meas_MID00461_FID05805_sub_015_BC.dat → rawdata/sub-015_BC.dat
Moved .dat: meas_MID00449_FID05793_AdjCoilSens.dat → rawdata/meas_MID00449_FID05793_AdjCoilSens.dat
Moved and renamed: 015_fixed_dot-16_grid_T1w_2026-03-04_19h27.03.147.log → et/015_mreyetrack_4points_2026-03-04_19h27.03.147.log
Moved and renamed: 015_fixed_dot-16_grid_T1w_2026-03-04_19h27.03.147.EDF → et/015_mreyetrack_4points_2026-03-04_19h27.03.147.EDF
Moved and renamed: 015_fixed_dot-16_grid_T1w_2026-03-04_19h27.03.147.csv → et/015_mreyetrack_4points_2026-03-04_19h27.03.147.csv
Moved .dat: meas_MID00454_FID05798_sub_015_HC.dat → rawdata/sub-015_HC.dat
Moved .dat: meas_MID00450_FID05794_sub_015_T1wLIBRE.dat → rawdata/sub-015_T1wLIBRE.dat
Moved and renamed: 015

For 2.0 MR-Eye RR

In [ ]:
import os
import shutil
import re
from pathlib import Path

# Define the folder path
folder_path = "/media/debi/SanDisk1TB/Work/260218/sub-004"

# Extract the subject number from the folder name
subject_num = os.path.basename(folder_path).split('-')[1]  # e.g., "012" from "sub-012"

# Create the target folders
rawdata_folder = os.path.join(folder_path, 'rawdata')
dicom_folder = os.path.join(folder_path, 'dicom')
et_folder1 = os.path.join(folder_path, 'et_mreyetrack')
et_folder2 = os.path.join(folder_path, 'et_2p0mreye')

for folder in [rawdata_folder, dicom_folder, et_folder1, et_folder2]:
    os.makedirs(folder, exist_ok=True)
    print(f"Created/verified folder: {folder}")

# First, move directories ending with 'MR' to dicom folder
for item in os.listdir(folder_path):
    item_path = os.path.join(folder_path, item)
    if os.path.isdir(item_path) and item.endswith('MR'):
        # Skip if it's one of our created folders
        if item_path not in [rawdata_folder, dicom_folder, et_folder1, et_folder2]:
            dest_path = os.path.join(dicom_folder, item)
            shutil.move(item_path, dest_path)
            print(f"Moved directory: {item} → dicom/")

# Process files recursively
for root, dirs, files in os.walk(folder_path):
    for filename in files:
        file_path = os.path.join(root, filename)
        
        # Skip files in the newly created folders
        if any(file_path.startswith(skip_folder) for skip_folder in [rawdata_folder, dicom_folder, et_folder1, et_folder2]):
            continue
        
        new_filename = filename
        
        # Original Renaming Rules
        # Rule 1: Replace 'fixed_dot-16_grid_T1w' with 'mreyetrack_4points'
        if 'fixed_dot-16_grid_T1w' in filename:
            new_filename = filename.replace('fixed_dot-16_grid_T1w', 'mreyetrack_4points')
            # Replace leading three digits with subject number
            new_filename = re.sub(r'^\d{3}_', f'{subject_num}_', new_filename)
            dest_path = os.path.join(et_folder1, new_filename)
            shutil.move(file_path, dest_path)
            print(f"Moved and renamed: {filename} → et_mreyetrack/{new_filename}")

        # Rule 2: Move 'fixation_dots_T1weighted' to et_folder2
        if 'fixation_dots_T1weighted' in filename:
            # Replace leading n digits with subject number
            new_filename = re.sub(r'^\d+_', f'{subject_num}_', new_filename)
            dest_path = os.path.join(et_folder2, new_filename)
            shutil.move(file_path, dest_path)
            print(f"Moved and renamed: {filename} → et_2p0mreye/{new_filename}")
        
        # Rule 3: For .dat files, remove 'meas_MIDXXXXX_FIDXXXXX_' and add sub-YYY prefix
        if filename.endswith('.dat'):
            # Remove the pattern meas_MIDXXXXX_FIDXXXXX_
            new_filename = re.sub(r'meas_MID\d+_FID\d+_', '', new_filename)
            # Add subject prefix if not already present
            if not new_filename.startswith('sub-'):
                new_filename = f"sub-{subject_num}_{new_filename}"
        
        # File Organization Rules
        # Rule 4: Move .dat files to rawdata
        if filename.endswith('.dat'):
            dest_path = os.path.join(rawdata_folder, new_filename)
            shutil.move(file_path, dest_path)
            print(f"Moved .dat: {filename} → rawdata/{new_filename}")
        
print("\nFile organization and renaming completed!")


Created/verified folder: /media/debi/SanDisk1TB/Work/260218/sub-004/rawdata
Created/verified folder: /media/debi/SanDisk1TB/Work/260218/sub-004/dicom
Created/verified folder: /media/debi/SanDisk1TB/Work/260218/sub-004/et_mreyetrack
Created/verified folder: /media/debi/SanDisk1TB/Work/260218/sub-004/et_2p0mreye
Moved directory: csTFL_mp-rage_1mm-iso_CP_acc4.6_5_MR → dicom/
Moved directory: t1_vibe_fs_tra_p2_0.5_6_MR → dicom/
Moved directory: t2_tse_tra_fs_1.5mm_orbites_DRB_7_MR → dicom/
Moved and renamed: 016_fixed_dot-16_grid_T1w_2026-02-18_18h47.05.744.csv → et_mreyetrack/004_mreyetrack_4points_2026-02-18_18h47.05.744.csv
Moved and renamed: 016_fixed_dot-16_grid_T1w_2026-02-18_18h47.05.744.EDF → et_mreyetrack/004_mreyetrack_4points_2026-02-18_18h47.05.744.EDF
Moved and renamed: 016_fixed_dot-16_grid_T1w_2026-02-18_18h47.05.744.log → et_mreyetrack/004_mreyetrack_4points_2026-02-18_18h47.05.744.log
Moved and renamed: 016_fixed_dot-16_grid_T1w_2026-02-18_18h47.05.744.psydat → et_mreyetra

Rename excluded subjects

In [4]:
import os

old_subject_num = '014'
new_subject_num = '013'
# base_folder = f"/home/debi/jaime/repos/MR-EyeTrack/data/study/sub-{old_subject_num}"
base_folder = f"/mnt/filer01/MatTechLab/jaime.barranco/MR-EyeTrack/data/study/sub-{old_subject_num}"

# Rename files and folders recursively, starting from deepest level
for root, dirs, files in os.walk(base_folder, topdown=False):
    # Rename files
    for filename in files:
        if old_subject_num in filename:
            old_path = os.path.join(root, filename)
            new_filename = filename.replace(f'sub-{old_subject_num}', f'sub-{new_subject_num}').replace(f'{old_subject_num}_', f'{new_subject_num}_')
            new_path = os.path.join(root, new_filename)
            os.rename(old_path, new_path)
            print(f"Renamed file: {filename} → {new_filename}")
    
    # Rename directories
    for dirname in dirs:
        if old_subject_num in dirname:
            old_path = os.path.join(root, dirname)
            new_dirname = dirname.replace(f'sub-{old_subject_num}', f'sub-{new_subject_num}')
            new_path = os.path.join(root, new_dirname)
            os.rename(old_path, new_path)
            print(f"Renamed folder: {dirname} → {new_dirname}")

# Finally, rename the base folder itself
# new_base_folder = f"/home/debi/jaime/repos/MR-EyeTrack/data/study/sub-{new_subject_num}"
new_base_folder = f"/mnt/filer01/MatTechLab/jaime.barranco/MR-EyeTrack/data/study/sub-{new_subject_num}"
os.rename(base_folder, new_base_folder)
print(f"Renamed base folder: sub-{old_subject_num} → sub-{new_subject_num}")

print("\nRenaming completed!")

Renamed file: sub-014_HC.dat → sub-013_HC.dat
Renamed file: sub-014_AdjCoilSens.dat → sub-013_AdjCoilSens.dat
Renamed file: sub-014_T1wLIBRE.dat → sub-013_T1wLIBRE.dat
Renamed file: sub-014_BC.dat → sub-013_BC.dat
Renamed file: sub-014.dcm → sub-013.dcm
Renamed file: 014_mreyetrack_4points_2026-02-25_15h52.01.541.EDF → 013_mreyetrack_4points_2026-02-25_15h52.01.541.EDF
Renamed file: 014_mreyetrack_4points_2026-02-25_15h52.01.541.log → 013_mreyetrack_4points_2026-02-25_15h52.01.541.log
Renamed file: 014_mreyetrack_4points_2026-02-25_15h52.01.541.csv → 013_mreyetrack_4points_2026-02-25_15h52.01.541.csv
Renamed file: 014_mreyetrack_4points_2026-02-25_15h52.01.541.psydat → 013_mreyetrack_4points_2026-02-25_15h52.01.541.psydat
Renamed base folder: sub-014 → sub-013

Renaming completed!
